# **1. 이안류 CCTV 데이터셋**
​AI Hub의 '이안류 CCTV 데이터'는 우리나라 주요 해수욕장(해운대, 송정, 대천, 중문, 낙산)에서 이안류 발생 여부와 위치를 모니터링하기 위해 구축된 인공지능 학습용 데이터셋입니다. 해수욕장 주변에 설치된 CCTV 영상을 이미지로 변환하여, 이안류 발생 여부와 위치를 가시화하는 모델 개발에 활용할 수 있습니다. 이 데이터셋은 이안류 탐지 및 예측 시스템 개발에 필수적인 자료를 제공하며, 해수욕객의 안전을 위한 응용 서비스 구성에 활용될 수 있습니다. ​이안류는 해안에서 먼 바다로 빠르게 이동하는 폭이 좁은 바닷물의 흐름으로, 기상 상태가 양호한 경우에도 나타나며, 얕은 곳에 있던 해수욕객을 순식간에 수심이 깊은 먼 바다로 이동시켜 인명사고를 유발할 수 있습니다. 따라서 이러한 데이터셋은 해수욕장 안전 관리 및 이안류 예측 모델 개발에 중요한 역할을 합니다.

아래 AIHub에서 이안류 CCTV 데이터셋을 다운로드 받습니다.

https://www.aihub.or.kr/aihubdata/data/view.do?currMenu=115&topMenu=100&aihubDataSe=data&dataSetSn=71297

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import json  # 라벨이 json이라
import shutil # 파일 관련 모듈(삭제, 복사 등)
import yaml
import random
from ultralytics import YOLO
from pathlib import Path

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


# 2. YOLO 데이터셋 만들기
### 1. JSON Bounding Box > YOLO Bounding Box
- 원본 JSON의 drawing에는 객체를 둘러싼 사각형의 점 좌표가 들어있음
- YOLO의 일반적인 객체 탐지 라벨 형식
    - 클래스 ID X센터 Y센터 너비 높이
    - 네 개의 좌표를 모두 0 ~ 1사이로 정규화

In [5]:
BASE_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/dataset")
DATA_DIR = BASE_ROOT / "이안류CCTV"
# VOC_ROOT = DATA_DIR / "VOC"
YOLO_ROOT = DATA_DIR / "TRAIN_이안류CCTV_DATASET"

# ANNOTATIONS_DIR = VOC_ROOT / "Annotations"
# IMAGES_DIR = VOC_ROOT / "JPGImages"

In [6]:
for sub in [
    "images/train", "images/val", "images/test",
    "labels/train", "labels/val", "labels/test"
]:
    (YOLO_ROOT/sub).mkdir(parents=True, exist_ok=True)

In [7]:
# Annotations에서 Object가 No, Yes 그리고 class가 0, 1로 되어 있음
VOC_CLASSES = [
    "Yes"
]

# 예측 클래스 명칭(문자열)을 고유한 번호표(ID) 값으로 매핑
CLASS_TO_ID = {name: idx for idx, name in enumerate(VOC_CLASSES)}
CLASS_TO_ID


{'Yes': 0}

In [8]:
# 이미지 좌상단/우하단 좌표를 YOLO 중심점 데이터로 연산
def voc_box_to_yolo(xmin, ymin, xmax, ymax, image_w, image_h):
    x_center = ((xmin + xmax)/2.0) / image_w
    y_center = ((ymin + ymax)/2.0) / image_h
    box_w = (xmax - xmin) / image_w
    box_h = (ymax - ymin) / image_h
    return x_center, y_center, box_w, box_h

In [9]:
def convert_voc_json(json_path, label_path):
    root = json.loads(Path(json_path).read_text(encoding="utf-8"))

    if root["annotations"]["object"] != "Yes":
        return

    resolution = root["image_info"]["resolution"]
    image_w, image_h = map(float, resolution.split(","))

    drawing = root["annotations"]["drawing"]

    lines = []

    for box in drawing:
        xs = [point[0] for point in box]
        ys = [point[1] for point in box]

        xmin, xmax = min(xs), max(xs)
        ymin, ymax = min(ys), max(ys)

        xmin = max(0.0, min(xmin, image_w))
        xmax = max(0.0, min(xmax, image_w))
        ymin = max(0.0, min(ymin, image_h))
        ymax = max(0.0, min(ymax, image_h))

        if xmax <= xmin or ymax <= ymin:
            continue

        x, y, w, h = voc_box_to_yolo(xmin, ymin, xmax, ymax, image_w, image_h)
        class_id = CLASS_TO_ID["Yes"]
        lines.append(f"{class_id} {x:.6f} {y:.6f} {w:.6f} {h:.6f}")

    label_path.parent.mkdir(parents=True, exist_ok=True)
    label_path.write_text("\n".join(lines), encoding="utf-8")

### 2. 원본 이미지와 JSON 라벨을 연결한 뒤 train/val/test 폴더로 나눔
- 같은 원본 영상에서 나온 프레임이 train과 test에 동시에 들어가면 성능이 실제보다 높게 보일 수 있음
- 파일명 앞부분을 그룹으로 사용해 가능한 하나의 같은 영상 계열이 한 세트에만 들어가도록 분리
- 데이터 수가 적으면 YOLO 성능이 낮아질 수 있음
    - 전체 이미지: 360장
    - 이안류 객체가 있는 이미지: 240장
    - 배경 이미지: 120장

> YOLO는 단순히 이미지를 외우는 것이 아니라, 려러 이미지에서 반복적으로 나타나는 시각적 특징을 학습하여 새로운 이미지에서도 객체를 찾아야 함. 단순한 파일 개수가 아니라 서로 다른 상황을 얼마나 다양하게 포함하고 있는지 중요

- cctv 연속 프레임에서는 무작위로 이미지를 섞어 나누는 방법을 사용하지 않음
    - 서로 몇 초 차이인 것의 동일한 프레임이 Train과 Test에 동시에 들어가면 모델은 Test 이미지를 처음 보는 것이 아니라, 학습 때 본 장면과 매우 유사한 정면을 다시 보는 것과 비슷해짐(데이터 누수)

- 전체 데이터셋을 확보할 수 있다면 아래 단위로 분리하는 것이 좋음
    - CCTV 카메라 각도
    - 해수욕장 위치
    - 날짜
    - 촬영 시간대

In [10]:
json_files = {}
image_files = {}

for original_split in ("train", "val"):
    split_root = DATA_DIR / original_split

    for p in (split_root / "라벨링데이터").rglob("*.json"):
        json_files[p.stem] = p

    for p in (split_root / "원천데이터").rglob("*.jpg"):
        image_files[p.stem] = p

In [11]:
groups = {}

for image_id in json_files:
    group_id = "_".join(image_id.split("_")[:3])
    groups.setdefault(group_id, []).append(image_id)

print("그룹 수:", len(groups))

그룹 수: 199


In [13]:
import random

rng = random.Random(2026)
items = list(json_files.items())
rng.shuffle(items)

yes_ids = []
no_ids = []

for image_id, json_path in items:
    label = json.loads(json_path.read_text(encoding="utf-8"))["annotations"]["object"]

    if label == "Yes" and len(yes_ids) < 240:
        yes_ids.append(image_id)
    elif label == "No" and len(no_ids) < 120:
        no_ids.append(image_id)

    if len(yes_ids) == 240 and len(no_ids) == 120:
        break

In [14]:
def prepare_split(split_name):

    out_image_dir = YOLO_ROOT / "images" / split_name
    out_label_dir = YOLO_ROOT / "labels" / split_name

    out_image_dir.mkdir(parents=True, exist_ok=True)
    out_label_dir.mkdir(parents=True, exist_ok = True)

    split_file = YOLO_ROOT / "ImageSets" / "Main" / f"{split_name}.txt"
    image_ids = [x.strip() for x in split_file.read_text(encoding="utf-8").splitlines() if x.strip()]

    for image_id in image_ids:
        src_image = image_files[image_id]
        src_json = json_files[image_id]

        shutil.copy2(src_image, out_image_dir / src_image.name)
        convert_voc_json(src_json, out_label_dir / f"{image_id}.txt")
    print(f'{split_name}: {len(image_ids):,} images prepared')

In [17]:
split_ids = {
    "train": yes_ids[:168] + no_ids[:84],
    "val": yes_ids[168:204] + no_ids[84:102],
    "test": yes_ids[204:240] + no_ids[102:120],
}

split_dir = YOLO_ROOT / "ImageSets" / "Main"
split_dir.mkdir(parents=True, exist_ok=True)

for split_name, image_ids in split_ids.items():
    rng.shuffle(image_ids)
    (split_dir / f"{split_name}.txt").write_text(
        "\n".join(image_ids), encoding="utf-8"
    )
    print(split_name, len(image_ids))

train 252
val 54
test 54


In [18]:
print(len(yes_ids), len(no_ids))
print((YOLO_ROOT / "ImageSets" / "Main" / "train.txt").exists())

240 120
True


In [19]:
prepare_split("train")
prepare_split("val")
prepare_split("test")

train: 252 images prepared
val: 54 images prepared
test: 54 images prepared


In [20]:
custom_voc_yaml = YOLO_ROOT / "custom_voc.yaml"

data_config = {
    "path": str(YOLO_ROOT.resolve()),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {i: name for i, name in enumerate(VOC_CLASSES)}
}

with open(custom_voc_yaml, "w", encoding="utf-8") as f:
    yaml.safe_dump(data_config, f, allow_unicode=True, sort_keys=False) # 알파벳 순으로 정렬할 필요가 없어서 False

print("생성된 YAML 경로: ", custom_voc_yaml)
print(custom_voc_yaml.read_text(encoding='utf-8'))

생성된 YAML 경로:  /content/drive/MyDrive/Colab Notebooks/dataset/이안류CCTV/TRAIN_이안류CCTV_DATASET/custom_voc.yaml
path: /content/drive/MyDrive/Colab Notebooks/dataset/이안류CCTV/TRAIN_이안류CCTV_DATASET
train: images/train
val: images/val
test: images/test
names:
  0: 'Yes'



In [21]:
model = YOLO("yolo11s.pt") # pt는 모델 아키텍처까지 있고 pth는 가중치만 있음
model

YOLO(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C3k2(
        (cv1): Conv(
          (conv): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(96, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(128, eps=0.001, momentum=0.03, affine=True, track_runnin

In [22]:
results = model.train(
    data = str(custom_voc_yaml),
    epochs = 10,
    imgsz = 640,
    batch = 16,
    # device = DEVICE,
    workers = 8,
    project = "/content/drive/MyDrive/Colab Notebooks/train_result/이안류",
    name = "이안_yolo11s",
    exist_ok = True,
)

Ultralytics 8.4.163 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Colab Notebooks/dataset/이안류CCTV/TRAIN_이안류CCTV_DATASET/custom_voc.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_s

In [23]:
best_path = Path("/content/drive/MyDrive/Colab Notebooks/train_result/이안류/이안_yolo11s/weights/best.pt")
model = YOLO(str(best_path))

In [31]:
val_results = model.val(
    data=str(custom_voc_yaml),
    split="val",
    imgsz=640,
    batch=16,
    workers=8,
    project="/content/drive/MyDrive/Colab Notebooks/train_result/이안류",
    name="val",
    exist_ok=True
)

Ultralytics 8.4.163 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11s summary (fused): 100 layers, 9,413,187 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access ✅ (ping: 0.5±0.1 ms, read: 116.9±20.6 MB/s, size: 142.7 KB)
val: Scanning /content/drive/MyDrive/Colab Notebooks/dataset/이안류CCTV/TRAIN_이안류CCTV_DATASET/labels/val.cache... 36 images, 18 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 54/54 11.3Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 2.1it/s 1.9s
                   all         54         49       0.26       0.51       0.25     0.0853
Speed: 4.2ms preprocess, 7.1ms inference, 0.0ms loss, 2.9ms postprocess per image
Results saved to /content/drive/MyDrive/Colab Notebooks/train_result/이안류/val


In [25]:
print('mAP50: ', val_results.box.map50)
print('mAP50-95: ', val_results.box.map)
print('mAP75: ', val_results.box.map75)

mAP50:  0.2496139992262073
mAP50-95:  0.08528598671317064
mAP75:  0.016842469602541536


In [30]:
val_images = list((YOLO_ROOT / "images" / "val").glob("*.jpg"))
sample_images = random.Random(2026).sample(val_images, 5)

pred_results = model.predict(
    source=[str(p) for p in sample_images],
    imgsz=640,
    conf=0.30,
    save=True,
    project="/content/drive/MyDrive/Colab Notebooks/train_result/이안류",
    name="test_images_predict",
    exist_ok=True
)

print(f"예측 이미지 수: {len(pred_results)}")


0: 384x640 1 Yes, 8.0ms
1: 384x640 (no detections), 8.0ms
2: 384x640 (no detections), 8.0ms
3: 384x640 (no detections), 8.0ms
4: 384x640 (no detections), 8.0ms
Speed: 2.3ms preprocess, 8.0ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)
Results saved to /content/drive/MyDrive/Colab Notebooks/train_result/이안류/test_images_predict
예측 이미지 수: 5


In [29]:
test_dir = YOLO_ROOT / "images" / "test"

pred_results = model.predict(
    source=str(test_dir),
    imgsz=640,
    conf=0.20,
    save=True,
    project="/content/drive/MyDrive/Colab Notebooks/train_result/이안류",
    name="test_images_predict_all",
    exist_ok=True
)

print(f"예측 이미지 수: {len(pred_results)}")


image 1/54 /content/drive/MyDrive/Colab Notebooks/dataset/이안류CCTV/TRAIN_이안류CCTV_DATASET/images/test/DC_MUDCH_20190705_134305.jpg: 384x640 (no detections), 22.7ms
image 2/54 /content/drive/MyDrive/Colab Notebooks/dataset/이안류CCTV/TRAIN_이안류CCTV_DATASET/images/test/DC_MUDCH_20190706_091100.jpg: 384x640 (no detections), 14.2ms
image 3/54 /content/drive/MyDrive/Colab Notebooks/dataset/이안류CCTV/TRAIN_이안류CCTV_DATASET/images/test/DC_MUDCH_20190706_091329.jpg: 384x640 (no detections), 13.8ms
image 4/54 /content/drive/MyDrive/Colab Notebooks/dataset/이안류CCTV/TRAIN_이안류CCTV_DATASET/images/test/DC_MUDCH_20200714_081655.jpg: 384x640 (no detections), 16.6ms
image 5/54 /content/drive/MyDrive/Colab Notebooks/dataset/이안류CCTV/TRAIN_이안류CCTV_DATASET/images/test/DC_MUDCH_20200724_073532.jpg: 384x640 (no detections), 14.3ms
image 6/54 /content/drive/MyDrive/Colab Notebooks/dataset/이안류CCTV/TRAIN_이안류CCTV_DATASET/images/test/DC_MUDCH_20200724_080813.jpg: 384x640 (no detections), 15.0ms
image 7/54 /content/drive/M

In [ ]:
import cv2
from pathlib import Path

compare_dir = Path(
    "/content/drive/MyDrive/Colab Notebooks/train_result/이안류/test_compare"
)
compare_dir.mkdir(parents=True, exist_ok=True)

for result in pred_results:
    image = result.orig_img.copy()
    image_h, image_w = image.shape[:2]

    # 정답 라벨: 빨간색
    label_path = YOLO_ROOT / "labels" / "test" / f"{Path(result.path).stem}.txt"
    if label_path.exists():
        for line in label_path.read_text(encoding="utf-8").splitlines():
            _, x, y, w, h = map(float, line.split())
            x1 = int((x - w / 2) * image_w)
            y1 = int((y - h / 2) * image_h)
            x2 = int((x + w / 2) * image_w)
            y2 = int((y + h / 2) * image_h)
            cv2.rectangle(image, (x1, y1), (x2, y2), (0, 0, 255), 3)

    # 모델 예측: 파란색
    for x1, y1, x2, y2 in result.boxes.xyxy.cpu().tolist():
        cv2.rectangle(
            image, (int(x1), int(y1)), (int(x2), int(y2)),
            (255, 0, 0), 3
        )

    cv2.imwrite(str(compare_dir / Path(result.path).name), image)

print(f"비교 이미지 {len(pred_results)}장 저장: {compare_dir}")